# Model Training and Validation

**EXSU 500 — Prostate Cancer Stage Classification**

This notebook trains and validates two supervised learning algorithms:
- Logistic Regression
- Random Forest

Three genomic feature sets are evaluated:
- Somatic mutations
- Copy-number alterations (CNAs)
- Combined mutations and CNAs

All models use the same predefined patient-level training and validation splits. The held-out test set is reserved for final evaluation.

## 1. Data Loading

Load the three genomic feature matrices and patient-level split assignments generated in `02_split_preprocessing.ipynb`.

The dataset contains 1,967 patients:
- Mutation-only: 341 features
- CNA-only: 340 features
- Combined: 681 features

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

DATA_DIR = Path("../data/processed")
RANDOM_STATE = 42

# Load the three feature sets prepared in notebook 02
feature_files = {
    "Mutation": "mutation_feature_matrix.csv",
    "CNA": "cna_feature_matrix_common.csv",
    "Combined": "combined_feature_matrix.csv",
}

feature_sets = {
    name: pd.read_csv(DATA_DIR / filename, index_col="PATIENT_ID")
    for name, filename in feature_files.items()
}

splits = pd.read_csv(DATA_DIR / "patient_splits.csv")

print("Feature matrix dimensions:")
for name, X in feature_sets.items():
    print(f"  {name}: {X.shape}")

print("\nPatients per split:")
print(splits["split"].value_counts())

Feature matrix dimensions:
  Mutation: (1967, 341)
  CNA: (1967, 340)
  Combined: (1967, 681)

Patients per split:
split
train         1376
test           296
validation     295
Name: count, dtype: int64


## 2. Data Validation

Verify that genomic features and patient labels are correctly aligned.

Checks include:
- Unique patient identifiers
- Consistent patients across all feature sets
- Valid binary stage labels
- No missing or non-finite feature values
- No overlap between training and validation patients

The predefined 70/15/15 split is preserved.

In [2]:
required_columns = {"PATIENT_ID", "stage_binary", "split"}

assert required_columns.issubset(splits.columns)
assert splits["PATIENT_ID"].is_unique
assert splits["stage_binary"].isin([0, 1]).all()
assert set(splits["split"]) == {"train", "validation", "test"}

train_ids = splits.loc[
    splits["split"] == "train", "PATIENT_ID"
].tolist()

val_ids = splits.loc[
    splits["split"] == "validation", "PATIENT_ID"
].tolist()

assert set(train_ids).isdisjoint(val_ids)

y_by_patient = splits.set_index("PATIENT_ID")["stage_binary"]

y_train = y_by_patient.loc[train_ids]
y_val = y_by_patient.loc[val_ids]

for name, X in feature_sets.items():
    assert X.index.is_unique
    assert X.columns.is_unique
    assert set(X.index) == set(y_by_patient.index)
    assert np.isfinite(X.to_numpy(dtype=float)).all()

assert (
    feature_sets["Mutation"].shape[1]
    + feature_sets["CNA"].shape[1]
    == feature_sets["Combined"].shape[1]
)

print("PASS: Feature sets and patient splits validated.")
print(f"Training patients: {len(train_ids)}")
print(f"Validation patients: {len(val_ids)}")

PASS: Feature sets and patient splits validated.
Training patients: 1376
Validation patients: 295


## 3. Model Configuration

Two classification algorithms are evaluated:

**Logistic Regression**
- Linear classification baseline
- Standardization of genomic features
- Regularization to reduce overfitting

**Random Forest**
- Nonlinear ensemble classifier
- Captures complex relationships between genomic features
- Does not require feature scaling

Constant features are removed using VarianceThreshold.

All preprocessing steps are fitted using training data only to prevent information leakage.

In [3]:
def build_model(model_name):
    steps = [
        ("nonconstant", VarianceThreshold(threshold=0.0))
    ]

    if model_name == "Logistic Regression":
        steps += [
            ("scale", StandardScaler()),
            ("classifier", LogisticRegression(
                C=1.0,
                max_iter=3000,
                solver="liblinear",
                random_state=RANDOM_STATE,
            )),
        ]

    elif model_name == "Random Forest":
        steps += [
            ("classifier", RandomForestClassifier(
                n_estimators=300,
                min_samples_leaf=2,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )),
        ]

    else:
        raise ValueError(f"Unknown model: {model_name}")

    return Pipeline(steps)


model_names = [
    "Logistic Regression",
    "Random Forest"
]

## 4. Model Training and Validation

Train both algorithms on each genomic feature set, producing six model configurations.

Models are evaluated on the validation set using:
- AUROC: Discrimination between Stage I–III and Stage IV
- Average Precision (AP): Summary of precision-recall performance

Results are compared across all six configurations.

Validation metrics and prediction probabilities are saved for subsequent analysis. The held-out test set is not used during this stage.

In [4]:
results = []
trained_models = {}

validation_predictions = pd.DataFrame({
    "PATIENT_ID": val_ids,
    "stage_binary": y_val.to_numpy(),
})

for feature_name, X in feature_sets.items():
    X_train = X.loc[train_ids]
    X_val = X.loc[val_ids]

    for model_name in model_names:
        pipeline = build_model(model_name)
        pipeline.fit(X_train, y_train)

        val_prob = pipeline.predict_proba(X_val)[:, 1]

        auroc = roc_auc_score(y_val, val_prob)
        ap = average_precision_score(y_val, val_prob)

        trained_models[(feature_name, model_name)] = pipeline

        results.append({
            "Feature Set": feature_name,
            "Model": model_name,
            "Features Used": int(
                pipeline.named_steps["nonconstant"]
                .get_support().sum()
            ),
            "Validation AUROC": auroc,
            "Validation AP": ap,
        })

        validation_predictions[
            f"{feature_name}__{model_name}"
        ] = val_prob

        print(
            f"{feature_name:8} | {model_name:20} "
            f"| AUROC={auroc:.3f} | AP={ap:.3f}"
        )

results_df = pd.DataFrame(results).sort_values(
    ["Validation AUROC", "Validation AP"],
    ascending=False
).reset_index(drop=True)

results_df.to_csv(
    DATA_DIR / "validation_model_comparison.csv",
    index=False
)

validation_predictions.to_csv(
    DATA_DIR / "validation_predictions.csv",
    index=False
)

print("\nValidation comparison:")
print(results_df.to_string(
    index=False,
    float_format=lambda x: f"{x:.3f}"
))

print("\nValidation results saved.")

Mutation | Logistic Regression  | AUROC=0.529 | AP=0.563
Mutation | Random Forest        | AUROC=0.572 | AP=0.558
CNA      | Logistic Regression  | AUROC=0.536 | AP=0.548
CNA      | Random Forest        | AUROC=0.500 | AP=0.538
Combined | Logistic Regression  | AUROC=0.555 | AP=0.581
Combined | Random Forest        | AUROC=0.609 | AP=0.601

Validation comparison:
Feature Set               Model  Features Used  Validation AUROC  Validation AP
   Combined       Random Forest            615             0.609          0.601
   Mutation       Random Forest            324             0.572          0.558
   Combined Logistic Regression            615             0.555          0.581
        CNA Logistic Regression            291             0.536          0.548
   Mutation Logistic Regression            324             0.529          0.563
        CNA       Random Forest            291             0.500          0.538

Validation results saved.


## 5. Hyperparameter Tuning

A limited hyperparameter search is conducted using validation AUROC.

Logistic regression regularization and random forest complexity
are adjusted while retaining the original patient-level splits.

The held-out test set remains unused.

In [5]:
# Small, predefined hyperparameter search
tuning_results = []

for feature_name, X in feature_sets.items():
    X_train = X.loc[train_ids]
    X_val = X.loc[val_ids]

    # Logistic regression: regularization strength
    for C in [0.01, 0.1, 1.0, 10.0]:
        model = build_model("Logistic Regression")
        model.set_params(classifier__C=C)

        model.fit(X_train, y_train)
        probabilities = model.predict_proba(X_val)[:, 1]

        tuning_results.append({
            "Feature Set": feature_name,
            "Model": "Logistic Regression",
            "Parameters": f"C={C}",
            "AUROC": roc_auc_score(y_val, probabilities),
            "AP": average_precision_score(y_val, probabilities),
        })

    # Random forest: tree complexity
    for depth, leaf in [
        (None, 2),
        (None, 5),
        (10, 5),
        (10, 10),
    ]:
        model = build_model("Random Forest")
        model.set_params(
            classifier__max_depth=depth,
            classifier__min_samples_leaf=leaf,
        )

        model.fit(X_train, y_train)
        probabilities = model.predict_proba(X_val)[:, 1]

        tuning_results.append({
            "Feature Set": feature_name,
            "Model": "Random Forest",
            "Parameters": f"depth={depth}, leaf={leaf}",
            "AUROC": roc_auc_score(y_val, probabilities),
            "AP": average_precision_score(y_val, probabilities),
        })

tuning_df = pd.DataFrame(tuning_results)

# Select best configuration for each model and feature set
best_tuning = (
    tuning_df.sort_values("AUROC", ascending=False)
    .drop_duplicates(["Feature Set", "Model"])
    .sort_values("AUROC", ascending=False)
)

display(best_tuning)

tuning_df.to_csv(
    DATA_DIR / "hyperparameter_tuning.csv",
    index=False
)

,Feature Set,Model,Parameters,AUROC,AP
21,Combined,Random Forest,"depth=None, leaf=5",0.612163,0.606048
6,Mutation,Random Forest,"depth=10, leaf=5",0.599245,0.574690
16,Combined,Logistic Regression,C=0.01,0.557935,0.581806
0,Mutation,Logistic Regression,C=0.01,0.551073,0.574281
11,CNA,Logistic Regression,C=10.0,0.542484,0.551342
14,CNA,Random Forest,"depth=10, leaf=5",0.527356,0.557650


## 6. Final Model Selection and Refitting

The best hyperparameters for each of the six model configurations
are selected using validation AUROC.

Each model is then refitted using the combined training and
validation datasets.

The held-out test set remains unused and will be evaluated
only in `04_model_evaluation.ipynb`.

In [6]:
import joblib

MODEL_DIR = DATA_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Selected hyperparameters from validation tuning
selected_params = {
    ("Mutation", "Logistic Regression"): {
        "classifier__C": 0.01
    },
    ("Mutation", "Random Forest"): {
        "classifier__max_depth": 10,
        "classifier__min_samples_leaf": 5
    },
    ("CNA", "Logistic Regression"): {
        "classifier__C": 10.0
    },
    ("CNA", "Random Forest"): {
        "classifier__max_depth": 10,
        "classifier__min_samples_leaf": 5
    },
    ("Combined", "Logistic Regression"): {
        "classifier__C": 0.01
    },
    ("Combined", "Random Forest"): {
        "classifier__max_depth": None,
        "classifier__min_samples_leaf": 5
    }
}

# Combine training and validation patients
train_val_ids = splits.loc[
    splits["split"].isin(["train", "validation"]),
    "PATIENT_ID"
].tolist()

y_all = splits.set_index("PATIENT_ID")["stage_binary"]
y_train_val = y_all.loc[train_val_ids]

assert len(train_val_ids) == 1671
assert set(train_val_ids).isdisjoint(
    set(splits.loc[splits["split"] == "test", "PATIENT_ID"])
)

# Refit and save all six models
for (feature_name, model_name), params in selected_params.items():

    X_train_val = feature_sets[feature_name].loc[train_val_ids]

    model = build_model(model_name)
    model.set_params(**params)
    model.fit(X_train_val, y_train_val)

    filename = (
        f"{feature_name.lower()}_"
        f"{model_name.lower().replace(' ', '_')}.joblib"
    )

    joblib.dump(model, MODEL_DIR / filename)

    print(f"Saved: {filename}")

print("\nAll six final models saved.")

Saved: mutation_logistic_regression.joblib
Saved: mutation_random_forest.joblib
Saved: cna_logistic_regression.joblib
Saved: cna_random_forest.joblib
Saved: combined_logistic_regression.joblib
Saved: combined_random_forest.joblib

All six final models saved.
